# Лабораторная 06. Shuffle join vs broadcast join

Цель: сравнить SortMergeJoin и BroadcastHashJoin.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

spark = (SparkSession.builder.appName('lab-06-broadcast-join').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Часть 1. Отключаем auto broadcast

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
shuffle_join = orders.join(customers, 'customer_id')
shuffle_join.explain('formatted')
shuffle_join.count()

## Часть 2. Явный broadcast маленького справочника

In [ ]:
broadcast_join = orders.join(broadcast(customers), 'customer_id')
broadcast_join.explain('formatted')
broadcast_join.count()

Сравните планы:

| Вопрос | Ответ |
|---|---|
| Какой join был в первом плане? | ... |
| Какой join был во втором плане? | ... |
| Где исчез или уменьшился shuffle? | ... |
| Почему маленький справочник можно broadcast'ить? | ... |
| Когда broadcast может навредить? | ... |

Подсказка: ищите `SortMergeJoin`, `BroadcastHashJoin`, `Exchange`, `BroadcastExchange`.